# Thermal Stack

Le pipeline est dans `thermal_stack.py`. Les donnees marche (spot, consommation 30 min, fuel cocktail) sont chargees via `data_loader.py`.

In [ ]:
import pandas as pd
from IPython.display import display

from thermal_stack import (
    ThermalStackConfig,
    build_thermal_stack,
    merit_order_for_areas,
    plot_merit_order_plotly,
    plot_regional_merit_order_plotly,
    identify_price_setters,
    summarize_price_setters,
    plot_price_setter_share_plotly,
)
import data_loader

In [ ]:
result = build_thermal_stack(ThermalStackConfig(delivery_month="2026-04", export_excel=False))
print(f"Delivery month: {result.delivery_month:%Y-%m}")
print(f"Thermal stack rows: {len(result.merit_order):,}")

## Merit Order

In [ ]:
region = "Tokyo"
mo_region = merit_order_for_areas(result, areas=region)
median_snapshot = identify_price_setters(mo_region, region).loc[lambda df: df.index.month == result.delivery_month.month]
median_demand = median_snapshot["demand_gw"].median()
median_spot = median_snapshot["spot_eur_mwh"].median()

fig = plot_merit_order_plotly(
    mo_region,
    region=region,
    period=result.delivery_month,
    demand_gw=median_demand,
    spot_price_eur_mwh=median_spot,
    title=f"Merit Order ({region}) - {result.delivery_month:%B %Y}",
)
fig.show()

In [ ]:
fig_regional = plot_regional_merit_order_plotly(result)
fig_regional.show()

## Marginal price setter

In [ ]:
spot = data_loader.load_jepx_spot()
df_30min = data_loader.load_df30min()
cocktail = data_loader.load_japan_fuel_cocktail()

regions = ["Tokyo", "Kansai"]
setter_tables = {}
setter_summaries = {}

for region in regions:
    mo_region = merit_order_for_areas(result, areas=region)
    setters = identify_price_setters(
        mo_region,
        region,
        spot=spot,
        df_30min=df_30min,
        fuel_cocktail=cocktail,
        start=result.delivery_month,
        end=result.delivery_month + pd.offsets.MonthEnd(0),
    )
    setter_tables[region] = setters
    setter_summaries[region] = summarize_price_setters(setters)
    display(setter_summaries[region])

In [ ]:
for region, summary in setter_summaries.items():
    fig_share = plot_price_setter_share_plotly(
        summary,
        region,
        title=f"Marginal fuel share - {region} - {result.delivery_month:%B %Y}",
    )
    fig_share.show()

In [ ]:
sample = setter_tables["Tokyo"].copy()
sample["hour"] = sample.index.hour
hourly_price_setter = (
    sample.groupby(["hour", "marginal_fuel_by_price"])
    .size()
    .unstack(fill_value=0)
    .div(sample.groupby("hour").size(), axis=0)
)
hourly_price_setter